# Single-subject pipeline

Set `SUBJECT_ID` and `CONDITION` in the next cell and run all cells.
You get: signal quality, ERP, epoch rejection stats, and classifier results.

Outputs are saved to `data/derived/`. Re-running a cell overwrites existing files.

In [ ]:
# ---- CONFIGURE HERE -------------------------------------------------------
SUBJECT_ID  = '31'   # bare id, e.g. '01' or 'pilot-self-day0'
CONDITION   = 'control'           # the recording to visualise at load time
CHANNEL     = 'Cz'               # channel for ERP plot
CLASSIFIER  = 'swlda'            # SWLDA is the official classifier of the paper
DATA_DIR    = 'data/raw'
DERIVED_DIR = 'data/derived'
# ---------------------------------------------------------------------------

In [ ]:
import sys
import os
from pathlib import Path

# Find repo root by walking up until we find config.yaml.
# This works whether Jupyter starts from notebooks/ or from the repo root.
_here = Path('.').resolve()
repo_root = next(
    (p for p in [_here, _here.parent, _here.parent.parent]
     if (p / 'config.yaml').exists()),
    _here,
)
os.chdir(repo_root)
sys.path.insert(0, str(repo_root))
print(f'Working directory: {Path.cwd()}')

%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 120

from analysis.loader import load_recording
from analysis.preprocess import preprocess_recording
from analysis.classifier import train_and_evaluate
from analysis.plots import (
    plot_erp, plot_psd, plot_rejection_summary,
    plot_confusion_matrix, plot_condition_accuracy, plot_subject_summary,
)
print('Imports OK')

## 1. Load

In [ ]:
rec = load_recording(SUBJECT_ID, CONDITION, data_dir=DATA_DIR, verbose_clock=True)

print(f'Subject:    sub-{rec.subject_id}')
print(f'Condition:  {rec.condition}')
print(f'Duration:   {rec.duration_s:.1f}s')
print(f'Markers:    {rec.n_epochs_planned}')
print(f'Clock fit:  m={rec.clock_fit[0]:.6f}  c={rec.clock_fit[1]*1000:.2f}ms')
print(f'EEG shape:  {rec.raw.get_data().shape}')

## 2. Signal quality — raw PSD

In [ ]:
fig = plot_psd(rec.raw, title=f'PSD — sub-{SUBJECT_ID}  {CONDITION}  (pre-filter)')
plt.show()

## 3. Preprocess — filter, epoch, baseline-correct, reject

In [ ]:
result = preprocess_recording(rec, save=True, derived_root=DERIVED_DIR)

log = result.rejection_log
print(f'Planned:           {log["n_planned"]}')
print(f'Boundary dropped:  {log["n_boundary_dropped"]}')
print(f'Amplitude dropped: {log["n_amplitude_dropped"]}')
print(f'Kept:              {log["n_kept"]}  ({(1-log["rejection_rate"])*100:.1f}%)')

### Vigilance check for this condition

Per the prerec document at §4: a condition is flagged if 2+ of its 3 sub-blocks deviate > 30% from
expected count. Please run the cell to check if the subject can be included in the analysis

In [ ]:
import json
from pathlib import Path

COUNT_THRESHOLD     = 0.30   # per-sub-block target-count discrepancy
SUBBLOCKS_TO_FLAG   = 2      # failing sub-blocks needed to flag a condition  (§4 crit 2, part 1)
CONDITIONS_TO_EXCL  = 2      # flagged conditions needed to exclude subject   (§4 crit 2, part 2)

CONDITIONS = ['control', 'acoustic', 'emi', 'chewing']

flagged_conditions = []
missing_conditions = []

for cond in CONDITIONS:
    sess_path = (
        Path(DATA_DIR) / f'sub-{SUBJECT_ID}'
        / f'sub-{SUBJECT_ID}_cond-{cond}_session.json'
    )

    print(f'\n=== {cond} ===')

    if not sess_path.exists():
        print('  (no session file — condition not yet recorded)')
        missing_conditions.append(cond)
        continue

    with open(sess_path) as f:
        blocks = json.load(f)['reported_counts']

    failed_blocks = []
    for blk in blocks:
        exp, rep = blk['expected_count'], blk['reported_count']
        dev = abs(rep - exp) / exp if exp else 0.0
        fail = dev > COUNT_THRESHOLD
        if fail:
            failed_blocks.append(blk['sub_block_index'])
        print(
            f'  sb{blk["sub_block_index"]}  '
            f'expected={exp:<4} reported={rep:<4} dev={dev*100:5.1f}%  '
            f'{"FAIL" if fail else "ok"}'
        )

    if len(failed_blocks) >= SUBBLOCKS_TO_FLAG:
        flagged_conditions.append(cond)
        print(
            f'  --> {cond} FLAGGED: {len(failed_blocks)} sub-blocks exceed '
            f'{COUNT_THRESHOLD*100:.0f}% (sb {failed_blocks}).'
        )
    else:
        print(f'  --> {cond} ok ({len(failed_blocks)} failing sub-blocks).')

# ---- subject-level verdict (§4 criterion 2) ----
print('\n' + '=' * 44)
print(f'sub-{SUBJECT_ID} — vigilance / count-discrepancy verdict')
print('=' * 44)

if missing_conditions:
    print(f'NOTE: missing sessions for {missing_conditions} — verdict is provisional.')

print(f'Flagged conditions ({len(flagged_conditions)}): {flagged_conditions or "none"}')

if len(flagged_conditions) >= CONDITIONS_TO_EXCL:
    print(
        f'\nVERDICT: EXCLUDE — {len(flagged_conditions)} flagged conditions '
        f'(>= {CONDITIONS_TO_EXCL}); §4 criterion 2 met.'
    )
else:
    print(
        f'\nVERDICT: RETAIN — {len(flagged_conditions)} flagged condition(s) '
        f'(< {CONDITIONS_TO_EXCL}); §4 criterion 2 not met.'
    )

In [ ]:
fig = plot_rejection_summary(result.rejection_log)
plt.show()

## 4. ERP — target vs nontarget

In [ ]:
fig = plot_erp(result, channel=CHANNEL)
plt.show()

In [ ]:
# All channels side by side — useful for checking spatial distribution of P300
fig, axes = plt.subplots(2, 4, figsize=(16, 6), sharey=True)
for ax, ch in zip(axes.flat, result.epochs.ch_names):
    plot_erp(result, channel=ch, ax=ax)
    ax.set_title(ch, fontsize=9)
    ax.set_xlabel('')
    ax.legend().remove()
fig.suptitle(f'ERP all channels — sub-{SUBJECT_ID}  {CONDITION}', fontsize=12)
fig.tight_layout()
plt.show()

## 5. Preprocess all four conditions

The classifier trains on control, so we need all four preprocessed before running it.

In [ ]:
ALL_CONDITIONS = ['control', 'chewing', 'emi', 'acoustic']
preprocess_results = {}

for cond in ALL_CONDITIONS:
    print(f'Preprocessing {cond}...')
    r = load_recording(SUBJECT_ID, cond, data_dir=DATA_DIR)
    pr = preprocess_recording(r, save=True, derived_root=DERIVED_DIR)
    preprocess_results[cond] = pr
    log = pr.rejection_log
    print(f'  kept {log["n_kept"]} / {log["n_planned"]}  '
          f'({(1-log["rejection_rate"])*100:.1f}% kept)')

print('\nAll conditions preprocessed and saved.')

## 6. Classifier — train on control, test on all

In [ ]:
clf_result = train_and_evaluate(
    subject_id=SUBJECT_ID,
    classifier_type=CLASSIFIER,
    derived_root=DERIVED_DIR,
    save=True,
)

print(f'Classifier:    {clf_result.classifier_type}')
print(f'Features used: {clf_result.n_features}')
print(f'Train epochs:  {clf_result.n_train_epochs} '
      f'(target={clf_result.train_target_count}, '
      f'nontarget={clf_result.train_nontarget_count})')
print()
for cond, metrics in clf_result.per_condition.items():
    print(f'  {cond:<20} bal.acc={metrics["balanced_accuracy"]*100:.1f}%  '
          f'n={metrics["n_epochs"]}')

In [ ]:
import json
from pathlib import Path

# Reload from disk so plot functions get the plain dict (not dataclass)
results_path = Path(DERIVED_DIR) / 'classifier-v2' / f'sub-{SUBJECT_ID}' / f'sub-{SUBJECT_ID}_results.json'
with open(results_path) as f:
    result_dict = json.load(f)

fig = plot_condition_accuracy(result_dict)
plt.show()

In [ ]:
# Confusion matrices for all conditions
conditions = list(result_dict['per_condition'].keys())
fig, axes = plt.subplots(1, len(conditions), figsize=(5 * len(conditions), 4))
for ax, cond in zip(axes, conditions):
    plot_confusion_matrix(result_dict['per_condition'], cond, ax=ax)
fig.suptitle(f'Confusion matrices — sub-{SUBJECT_ID}  [{CLASSIFIER}]', fontsize=12)
fig.tight_layout()
plt.show()

## 7. One-page summary

In [ ]:
# Uses the CONDITION recording preprocessed above (step 3)
fig = plot_subject_summary(preprocess_results[CONDITION], result_dict, channel=CHANNEL)
plt.show()

# Save it
out_path = Path(DERIVED_DIR) / f'sub-{SUBJECT_ID}_summary.png'
fig.savefig(out_path, dpi=150, bbox_inches='tight')
print(f'Saved: {out_path}')